In [1]:
import pandas as pd
import awswrangler as awr
import openpyxl
import shutil
import datetime as dt
import pyautogui
import time

In [2]:
def format_type(df):
    for col in df.select_dtypes(include=['string']).columns:
        df[col] = df[col].str.upper()
    return df

In [3]:
# EXTRAINDO DADOS DE ATIVOS

query_ativos = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_ATIVOS.sql"
with open(query_ativos, 'r', encoding='utf-8') as file:
    sql_ativos = file.read()   

df_ativos =awr.athena.read_sql_query(
    sql=sql_ativos,database='silver'
)

In [4]:
df_ativos = df_ativos.drop(columns=['rn'])

In [5]:
df_ativos = (
    df_ativos
    .sort_values(by=['chassi', 'inicio_vig', 'data_ativacao'], ascending=[True, False, False])
    .drop_duplicates(subset=['chassi'], keep='first')
)

In [6]:
# EXTRAINDO DADOS DE CANCELAMENTOS INTEGRAIS (CONJUNTO)

query_cancelados_integrais = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_CANCELAMENTOS_INTEGRAIS.sql"
with open(query_cancelados_integrais, 'r', encoding='utf-8') as file:
    sql_cancelados_integrais = file.read()   

df_cancelamentos_integrais =awr.athena.read_sql_query(
    sql=sql_cancelados_integrais, database='silver'
)

In [7]:
# EXTRAINDO DADOS DE CANCELAMENTOS PARCIAIS (CONJUNTO)

query_cancelados_parciais = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_CANCELAMENTOS_PARCIAIS.sql"
with open(query_cancelados_parciais, 'r', encoding='utf-8') as file:
    sql_cancelados_parciais = file.read()   

df_cancelamentos_parciais =awr.athena.read_sql_query(
    sql=sql_cancelados_parciais,database='silver'
)

In [8]:
format_type(df_ativos)
format_type(df_cancelamentos_integrais)
format_type(df_cancelamentos_parciais)

,placa,chassi,id_placa,id_veiculo,id_carroceria,matricula,conjunto,unidade,consultor,status,...,usuario_cancelamento,coverage_id,beneficio,status_beneficio,data_extracao,data_registro,data_ativacao,data_ativacao_beneficio,data_atualizacao,empresa
0,OAZ0E25,9EP021030B1006341,18796,0,18796,22230,15631,UNIDADE RONDONOPOLIS,RAPHAELLA SILVA,ATIVO,...,,96920,RASTREADOR REBOQUE/SEMIRREBOQUE,CANCELADO,2026-01-21,2025-08-26,2025-09-11,2025-09-11,2025-12-01,STCOOP
1,AYU0G87,9BSR6X200E3864540,4276,4276,<NA>,2494,13354,UNIDADE CURITIBA - INATIVA,AMILTON MARTUCCI,ATIVO,...,,83100,ASSESSORIA JURÍDICA,CANCELADO,2026-01-21,2025-01-28,2025-01-29,2025-01-29,2025-10-28,STCOOP
2,ASD7669,9ADG12439AM299596,16159,0,16159,3600,14276,UNIDADE ITAJAI,JOSÉ ELIAS DOS SANTOS FILHO,ATIVO,...,,88679,RASTREADOR REBOQUE/SEMIRREBOQUE,CANCELADO,2026-01-21,2025-04-25,2025-04-26,2025-04-26,2026-01-13,STCOOP
3,OAU3I79,93ZS3HUH0D8820730,23619,23619,<NA>,3623,13998,UNIDADE SINOP,QUEILA C. SANTOS,ATIVO,...,,87025,VIDROS,CANCELADO,2026-01-21,2025-03-28,2025-04-10,2025-04-10,2025-11-18,STCOOP
4,OAU3I79,93ZS3HUH0D8820730,23619,23619,<NA>,3623,13998,UNIDADE SINOP,QUEILA C. SANTOS,ATIVO,...,,87023,REPARAÇÃO A TERCEIROS,CANCELADO,2026-01-21,2025-03-28,2025-04-10,2025-04-10,2025-11-18,STCOOP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2480,QHL5B33,9BM958453FB000905,15997501,15997501,<NA>,7763,12938,MF - ROBSON GONCALVES DE QUEIROZ SILVA,MF - ROBSON GONCALVES,ATIVO,...,,98746,CASCO (VEÍCULO),CANCELADO,2026-01-21,2025-03-25,2025-04-04,2025-04-04,2025-12-02,VIAVANTE
2481,QJL8H90,97T0AN893KC005919,20368,0,20368,7771,13117,UNIDADE JOINVILLE,AUGUSTINHO TADEU PIGNATEL,ATIVO,...,,100431,CASCO (R/SR),CANCELADO,2026-01-21,2025-03-27,2025-03-31,2025-03-31,2025-08-29,VIAVANTE
2482,MLI3664,9BSR6X200D3836026,16296141,16296141,<NA>,7771,13040,UNIDADE JOINVILLE,AUGUSTINHO TADEU PIGNATEL,ATIVO,...,,99731,VIDROS,CANCELADO,2026-01-21,2025-03-26,2025-03-31,2025-03-31,2025-06-23,VIAVANTE
2483,MLI3664,9BSR6X200D3836026,16296141,16296141,<NA>,7771,13040,UNIDADE JOINVILLE,AUGUSTINHO TADEU PIGNATEL,ATIVO,...,,99729,ASSISTÊNCIA 24 HORAS,CANCELADO,2026-01-21,2025-03-26,2025-03-31,2025-03-31,2025-06-23,VIAVANTE


In [9]:

today_ts = pd.Timestamp.today().normalize()

if today_ts.weekday() == 0:  
    yesterday_ts = today_ts - dt.timedelta(days=3)
else:
    yesterday_ts = today_ts - dt.timedelta(days=1)
yesterday = yesterday_ts.strftime('%d-%m-%Y')


df_cancelamentos_integrais['data_cancelamento'] = pd.to_datetime(
	df_cancelamentos_integrais['data_cancelamento'], errors='coerce'
).dt.date

placas_canceladas_dia_anterior = df_cancelamentos_integrais.loc[
	df_cancelamentos_integrais['data_cancelamento'] == yesterday_ts.date(),
	'placa'
].unique().tolist()



In [10]:
df_cancelamentos_integrais['identificador'] = 'INTEGRAL'
df_cancelamentos_parciais['identificador'] = 'PARCIAL'

In [11]:
df_cancelamentos = pd.concat(
    [df_cancelamentos_integrais, df_cancelamentos_parciais], ignore_index=True
)

df_cancelamentos = df_cancelamentos.sort_values(
    by='data_cancelamento', ascending=False
).reset_index(drop=True)

df_cancelamentos.drop_duplicates(subset=['empresa', 'coverage_id', 'chassi'], keep='first')

,placa,chassi,id_placa,id_veiculo,id_carroceria,matricula,conjunto,unidade,consultor,status,...,beneficio,status_beneficio,data_extracao,data_registro,data_ativacao,data_ativacao_beneficio,data_cancelamento,empresa,identificador,data_atualizacao
0,CUE5H77,9BSG4X20093643255,195732,195732,<NA>,230,7716,UNIDADE LONDRINA,DAIANE CRISTINA VEIGA DA SILVA,FINALIZADO,...,CASCO (VEÍCULO),ATIVO,2026-01-21,2025-01-03,2025-01-04,2025-01-04,2026-01-21,VIAVANTE,INTEGRAL,NaT
1,GGR5A16,9BVRG20C0KE861038,14711,14711,<NA>,40,12972,UNIDADE SÃO PAULO,JOÃO HENRIQUE SILVA NEVES,FINALIZADO,...,CASCO (VEÍCULO),ATIVO,2026-01-21,2025-01-02,2025-01-05,2025-01-05,2026-01-21,STCOOP,INTEGRAL,NaT
2,RME8I63,9BM938142LS057960,16216102,16216102,<NA>,2288,11724,TRANSDESK DIGITAL LTDA,CRISLAINE LIMA DA LUZ,CANCELADO,...,REPARAÇÃO A TERCEIROS,CANCELADO,2026-01-21,2025-03-11,2025-03-11,2025-03-11,2026-01-20,VIAVANTE,INTEGRAL,NaT
3,RFO2J31,9ADG1243LMM467022,26906,0,26906,2288,18087,TRANSDESK DIGITAL LTDA,CRISLAINE LIMA DA LUZ,CANCELADO,...,CASCO (R/SR),CANCELADO,2026-01-21,2025-06-09,2025-06-10,2025-06-10,2026-01-20,VIAVANTE,INTEGRAL,NaT
4,SXH9B81,9539J8TH0SR204117,15928759,15928759,<NA>,4211,8387,FRANQUEADO - JOSÉ ELIAS,JOSÉ ELIAS DOS SANTOS FILHO,CANCELADO,...,ASSISTÊNCIA 24 HORAS,CANCELADO,2026-01-21,2025-01-16,2025-01-20,2025-01-20,2026-01-20,VIAVANTE,INTEGRAL,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37821,QHL5B33,9BM958453FB000905,15997501,15997501,<NA>,7763,12938,MF - ROBSON GONCALVES DE QUEIROZ SILVA,MF - ROBSON GONCALVES,ATIVO,...,CASCO (VEÍCULO),CANCELADO,2026-01-21,2025-03-25,2025-04-04,2025-04-04,NaN,VIAVANTE,PARCIAL,2025-12-02
37822,QJL8H90,97T0AN893KC005919,20368,0,20368,7771,13117,UNIDADE JOINVILLE,AUGUSTINHO TADEU PIGNATEL,ATIVO,...,CASCO (R/SR),CANCELADO,2026-01-21,2025-03-27,2025-03-31,2025-03-31,NaN,VIAVANTE,PARCIAL,2025-08-29
37823,MLI3664,9BSR6X200D3836026,16296141,16296141,<NA>,7771,13040,UNIDADE JOINVILLE,AUGUSTINHO TADEU PIGNATEL,ATIVO,...,VIDROS,CANCELADO,2026-01-21,2025-03-26,2025-03-31,2025-03-31,NaN,VIAVANTE,PARCIAL,2025-06-23
37824,MLI3664,9BSR6X200D3836026,16296141,16296141,<NA>,7771,13040,UNIDADE JOINVILLE,AUGUSTINHO TADEU PIGNATEL,ATIVO,...,ASSISTÊNCIA 24 HORAS,CANCELADO,2026-01-21,2025-03-26,2025-03-31,2025-03-31,NaN,VIAVANTE,PARCIAL,2025-06-23


In [12]:

df_ativos = df_ativos.sort_values(
    by=['inicio_vig', 'data_ativacao'], ascending=False
).reset_index(drop=True)

In [13]:
today = dt.date.today()
if today.weekday() == 0:  # Segunda-feira
    yesterday = today - dt.timedelta(days=3)
else:
    yesterday = today - dt.timedelta(days=1)

yesterday

datetime.date(2026, 1, 20)

In [14]:
ativados_seg = len(df_ativos[(df_ativos['empresa'] == 'SEGTRUCK') & (df_ativos['inicio_vig'] == yesterday)])
ativados_st = len(df_ativos[(df_ativos['empresa'] == 'STCOOP') & (df_ativos['inicio_vig'] == yesterday)])
ativados_viav = len(df_ativos[(df_ativos['empresa'] == 'VIAVANTE') & (df_ativos['inicio_vig'] == yesterday)])
ativados_tag = len(df_ativos[(df_ativos['empresa'] == 'TAG') & (df_ativos['inicio_vig'] == yesterday)])

cancelados_seg = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'SEGTRUCK') & (df_cancelamentos['data_cancelamento'] == yesterday)])
cancelados_st = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'STCOOP') & (df_cancelamentos['data_cancelamento'] == yesterday)])
cancelados_viav = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'VIAVANTE') & (df_cancelamentos['data_cancelamento'] == yesterday)])
cancelados_tag = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'TAG') & (df_cancelamentos['data_cancelamento'] == yesterday)])

In [26]:
yesterday

datetime.date(2026, 1, 20)

In [25]:
ativados_tag

53

In [15]:
# FUNÇÃO PARA LIMPAR DADOS DA PLANILHA
def clear_sheet(sheet):
    max_row = sheet.max_row
    if max_row > 1:
        sheet.delete_rows(1,max_row)

In [16]:


template = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\template\TEMPLATE_BASE_ATIVACOES_CANCELAMENTOS.xlsx"

wb = openpyxl.load_workbook(template)

w1 = wb['ATIVOS']
w2 = wb['CANCELAMENTOS']
w3 = wb['BASE']
w4 = wb['RELATORIO']

clear_sheet(w1)
clear_sheet(w2)




In [17]:
# Fill NaN values: numeric columns with 0, string columns with 'N/A'
numeric_cols = df_ativos.select_dtypes(include=['number']).columns
string_cols = df_ativos.select_dtypes(include=['object', 'string']).columns

df_ativos[numeric_cols] = df_ativos[numeric_cols].fillna(0)
df_ativos[string_cols] = df_ativos[string_cols].fillna('N/A')

In [18]:
# Fill NaN values: numeric columns with 0, string columns with 'N/A'
numeric_cols = df_cancelamentos.select_dtypes(include=['number']).columns
string_cols = df_cancelamentos.select_dtypes(include=['object', 'string']).columns

df_cancelamentos[numeric_cols] = df_cancelamentos[numeric_cols].fillna(0)
df_cancelamentos[string_cols] = df_cancelamentos[string_cols].fillna('N/A')

In [19]:
# Inserir cabeçalhos
for c_idx, col_name in enumerate(df_ativos.columns, start=1):
    w1.cell(row=1, column=c_idx, value=col_name)

# Inserir dados (a partir da linha 2)
if not df_ativos.empty:
    for r_idx, row in enumerate(df_ativos.values, start=2):
        for c_idx, value in enumerate(row, start=1):
            w1.cell(row=r_idx, column=c_idx, value=value)

In [20]:
# INSERIR CABEÇALHOS
for c_idx, col_name in enumerate(df_cancelamentos.columns, start=1):
    w2.cell(row=1, column=c_idx, value=col_name)

# ADICIONANDO OS DADOS NA ABA 'CANCELAMENTOS INTEGRAIS'
if  df_cancelamentos.empty == False:
    for r_idx, row in enumerate(df_cancelamentos.values, start=2):
        for c_idx, value in enumerate(row, start=1):
            w2.cell(row=r_idx, column=c_idx, value=value)

### criando base

In [21]:
# ENCONTRANDO PRIMEIRA LINHA VAZIA NA COLUNA B ABA 'BASE' 


first_empty_row = 1
for row in range(1, w3.max_row + 1):
    if w3.cell(row=row, column=2).value is None:
        first_empty_row = row
        break
else:
    first_empty_row = w3.max_row + 1

# PEGAR DATA ATUAL NA PLANILHA
data_atual_planilha = w3['A' + str(first_empty_row)].value

# Certifique-se de que 'yesterday' e 'data_atual_planilha' sejam ambos date
if isinstance(data_atual_planilha, pd.Timestamp):
    data_atual_planilha = data_atual_planilha.date()
elif hasattr(data_atual_planilha, 'date'):
    data_atual_planilha = data_atual_planilha.date()
elif isinstance(data_atual_planilha, dt.datetime):
    data_atual_planilha = data_atual_planilha.date()
elif isinstance(data_atual_planilha, str):
    try:
        data_atual_planilha = pd.to_datetime(data_atual_planilha).date()
    except Exception:
        pass

# Também garanta que 'yesterday' é date
if isinstance(yesterday, pd.Timestamp):
    yesterday_date = yesterday.date()
elif isinstance(yesterday, dt.datetime):
    yesterday_date = yesterday.date()
else:
    yesterday_date = yesterday

# VERIFICA SE DATA NA PLANILHA = DATA DE ONTEM
if data_atual_planilha == yesterday_date:
    # Filtra ativos até a data de ontem (inclusive)
    if "data_ativacao" in df_ativos.columns:
        # Supondo que data_ativacao já seja datetime ou string convertível
        data_ativacao_col = pd.to_datetime(df_ativos["data_ativacao"], errors='coerce')
        df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(yesterday_date)]
        qtd_ativos = df_ativos_filtrados['chassi'].nunique()
    else:
        qtd_ativos = df_ativos['chassi'].nunique()  # fallback: considera todos

    # Verifica se ontem foi sábado ou domingo
    dia_semana = yesterday_date.weekday()  # 5 = sábado, 6 = domingo
    if dia_semana == 5:
        w3['B' + str(first_empty_row)] = 'SÁBADO'
    elif dia_semana == 6:
        w3['B' + str(first_empty_row)] = 'DOMINGO'
    else:
        w3['B' + str(first_empty_row)] = qtd_ativos
    print('Registro de ativos preenchido na aba BASE!')
else:
    print('Datas não conferem, verifique o código! (data atual já pode estar preenchida)')

Datas não conferem, verifique o código! (data atual já pode estar preenchida)


### criando resumo

In [22]:
# INSERINDO INFORMAÇÕES DE ATIVADOS E CANCELADOS

w4['C6'] = ativados_seg
w4['C7'] = ativados_st
w4['C8'] = ativados_viav
w4['C9'] = ativados_tag

w4['D6'] = cancelados_seg
w4['D7'] = cancelados_st
w4['D8'] = cancelados_viav
w4['D9'] = cancelados_tag


### construindo relação de chassis por unidade

In [23]:
import os

# save workbook to the full file path
wb.save(template)

name_file = fr'RELATORIO_ATIVACOES_CANCELAMENTOS_{yesterday}.xlsx'

output_path = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\reports"
os.makedirs(output_path, exist_ok=True)
output_full_path = os.path.join(output_path, name_file)

# save workbook to the full file path
shutil.copy(template, output_full_path)

name_file_2 = fr'RELATORIO_ATIVACOES_CANCELAMENTOS.xlsx'

path_sharepoint = r"C:\Users\raphael.almeida\OneDrive - Grupo Unus\analise de dados - Arquivos em excel"
full_path_sharepoint = os.path.join(path_sharepoint, name_file_2)

# copy the saved file to the SharePoint folder (src, dst)
shutil.copy(output_full_path, full_path_sharepoint)

wb.close()